In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix,coo_matrix

# From Building the Hybrid Recommender:

Find more information about this in `HybridSongRecommender.ipynb`.

In [ ]:
music_info = pd.read_csv('Data/music_info_cleaned.csv')
users_history = pd.read_csv('Data/users_history_cleaned.csv')
music_info_metadata = pd.read_csv('Data/music_info_metadata.csv')

feature_columns = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 
                   'instrumentalness', 'liveness', 'valence', 'tempo']
numerical_features = music_info[feature_columns].values

In [ ]:
from sklearn.neighbors import NearestNeighbors


knn_index = NearestNeighbors(n_neighbors=50, algorithm='auto', metric='cosine').fit(numerical_features)

# Function to get nearest neighbors for a given track
def get_knn(knn_index, feature_matrix, item_id, n):
    distances, indices = knn_index.kneighbors(numerical_features[item_id:item_id+1], n_neighbors=n+1)
    # Return so it skips the first one (itself)
    return indices[0][1:], distances[0][1:]


In [4]:
train, test = train_test_split(users_history, test_size=0.2, random_state=42)

In [5]:
# For collaborative filtering (using users_history)
train['user_id'] = train['user_id'].astype('category')
train['track_id'] = train['track_id'].astype('category')

# Create mappings for user_id and track_id
cf_user_id_mapping = dict(enumerate(train['user_id'].cat.categories))
cf_track_id_mapping = dict(enumerate(train['track_id'].cat.categories))
cf_user_id_reverse_mapping = {v: k for k, v in cf_user_id_mapping.items()}
cf_track_id_reverse_mapping = {v: k for k, v in cf_track_id_mapping.items()}

# For content-based filtering (using music_info)
music_info['track_id'] = music_info['track_id'].astype('category')

cb_track_id_mapping = dict(enumerate(music_info['track_id'].cat.categories))
cb_track_id_reverse_mapping = {v: k for k, v in cb_track_id_mapping.items()}



In [6]:
from scipy.sparse import coo_matrix

# Fill missing playcounts with 0 before creating the sparse matrix
train_filled = train.copy()

user_item_sparse = coo_matrix((
    train_filled['playcount'],
    (train_filled['user_id'].cat.codes,
     train_filled['track_id'].cat.codes)
))

# Apply SVD on the Sparse Matrix
svd = TruncatedSVD(n_components=10, random_state=42)
user_factors = svd.fit_transform(user_item_sparse)
item_factors = svd.components_.T

In [7]:
# Converting user interactions into dictionary for train/test
# Inspired by Shivarov (2024) Kaggle notebook.
user_train_data = train.groupby('user_id', observed=True)['track_id'].apply(list).to_dict()
user_test_data = test.groupby('user_id', observed=True)['track_id'].apply(list).to_dict()

In [8]:
# One-hot encode Q1, Q2, Q3, Q4 for user
def mood_quadrant_to_vector(quadrant):
    q_map = {"Q1": [1,0,0,0], "Q2": [0,1,0,0], "Q3": [0,0,1,0], "Q4": [0,0,0,1]}
    return q_map[quadrant]


In [9]:
import random

# Find a random user from user history to test recommendations on
random_row = users_history.sample(n=1)
random_user_id = random_row['user_id'].values[0]
print(random_user_id)


1742426561b9dd2122f8af6ee7585f671a2dd335


In [10]:
# Get the tracks this user has listened to in the training data
user_tracks = train[train['user_id'] == random_user_id]['track_id'].tolist()

if not user_tracks:
    print(f"No listening history found for user {random_user_id}.")
else:
    # Pick a random track from their listening history
    user_track_id = random.choice(user_tracks)
    # Retrieve track name and artist from metadata
    track_info = music_info_metadata.loc[
        music_info_metadata['track_id'] == user_track_id, ['name', 'artist']
    ].iloc[0]
    
    random_track_name = track_info['name']
    random_track_artist = track_info['artist']

    print(f"User {random_user_id} listened to: {random_track_name} by {random_track_artist}")


User 1742426561b9dd2122f8af6ee7585f671a2dd335 listened to: Song for No One by Miike Snow


In [11]:
test_user_tracks = test.groupby('user_id')['track_id'].apply(list).to_dict()

# Evaluating:

* Splits data into train/test (80/20)
* Generates recommendations for test users using a hybrid approach *and* baselines (pure CF, pureCB)
* Computes average metrics across users at K=5

## Modified version of recommend_songs_hybrid for evaluation

In [ ]:
"""
    Modified version of recommend_songs_hybrid for evaluation (silent, returns scores without printing).

    beta_cf, beta_cb, beta_mood are weights for hybrid scoring. 
    If you want to run only CF, set beta_cf=1, beta_cb=0, beta_mood=0
    If you want to run only CB, set beta_cf=0, beta_cb=1, beta_mood=0
"""
def recommend_songs_hybrid_eval(user_id, track_name, user_item_matrix, user_factors, item_factors,
                                music_info_metadata, knn_index, numerical_features, user_mood=None,
                                n_recommendations=10, beta_cf=0.2, beta_cb=0.2, beta_mood=0.6): 
    # Dict for quick track metadata lookup
    track_metadata_dict = music_info_metadata.set_index('track_id')[['name', 'artist']].to_dict('index')
    
    # Validate user exists in mapping
    user_code = cf_user_id_reverse_mapping.get(user_id)
    if user_code is None:
        return []  # Return empty if invalid (silent for eval)
    
    # Validate track exists in metadata
    track_row = music_info_metadata[music_info_metadata['name'] == track_name]
    if track_row.empty:
        return []
        
    track_id = track_row.iloc[0]['track_id']
    track_code = cb_track_id_reverse_mapping.get(track_id)
    if track_code is None:
        return []

    # -----------------------------------------------------
    # Content-Based Filtering: Get similar tracks using kNN
    # -----------------------------------------------------
    similar_indices, distances_from_knn = get_knn(knn_index, numerical_features, track_code, n=25)
    cb_candidates = music_info_metadata.iloc[similar_indices][['track_id', 'name', 'artist']].copy()
    cb_recommended_tracks = cb_candidates['track_id'].tolist()
    # CB scores: Inverse of distances (higher similarity = higher score)
    cb_scores = {t: 1 - d for t, d in zip(cb_recommended_tracks, distances_from_knn)}

    # ------------------------------------------------------------------------------
    # Collaborative Filtering: Predict scores using dot product of user/item factors
    # ------------------------------------------------------------------------------
    cf_predictions = np.dot(user_factors[user_code, :], item_factors.T)
    cf_indices = np.argsort(cf_predictions)[::-1]
    cf_recommended_tracks = [cf_track_id_mapping[i] for i in cf_indices[:n_recommendations]]
    cf_scores = {
        t: np.dot(user_factors[user_code, :], item_factors[cf_track_id_reverse_mapping.get(t, 0), :])
        for t in cf_recommended_tracks
    }

    # Normalizating function to classify scores to [0,1]
    def min_max_normalize(scores):
        vals = np.array(list(scores.values()))
        if vals.max() == vals.min():
            return {k: 0.5 for k in scores}  # fallback if all scores equal
        return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in scores.items()}

    # Normalize CF and CB scores
    cf_scores_norm = min_max_normalize(cf_scores)
    cb_scores_norm = min_max_normalize(cb_scores)

    # -------------------
    # Mood-based scoring
    # -------------------
    mood_scores = {}
    for t in set(cf_scores_norm) | set(cb_scores_norm):
        mood_col = f'mood_{user_mood}'
        mood_scores[t] = music_info.loc[music_info['track_id'] == t, mood_col].values[0]
    # Normalize mood scores, like CF/CB
    mood_scores_norm = min_max_normalize(mood_scores)

    # Weighted hybrid scoring
    beta_cf = beta_cf
    beta_cb = beta_cb
    beta_mood = beta_mood
    
    hybrid_scores = {}
    for t in set(cf_scores_norm) | set(cb_scores_norm):
        hybrid_scores[t] = (
            beta_cf * cf_scores_norm.get(t, 0) +
            beta_cb * cb_scores_norm.get(t, 0) +
            beta_mood * mood_scores_norm.get(t, 0)
        )
    # Sort by hybrid score descending
    hybrid_sorted = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
    
    # Return top-K as (track_id, score) tuples
    return hybrid_sorted[:n_recommendations]




# Generate Recommendations for Test Users

In [ ]:
recommendations = {}        # Hybrid
recommendations_cf = {}     # Pure CF
recommendations_cb = {}     # Pure CB
count = 0
for user_id in test_user_tracks.keys():
    if count % 500 == 0:
        print(f"Processed {len(recommendations)} users")
    count += 1

    # Get user's training tracks
    user_train_tracks = user_train_data.get(user_id, [])
    if not user_train_tracks:
        continue  # Skip users without train data
    
    # Pick a random reference track form their train history
    ref_track_id = random.choice(user_train_tracks)
    ref_track_name = music_info_metadata[music_info_metadata['track_id'] == ref_track_id]['name'].iloc[0]
    
    # Randomly assign a mood quadrant for evaluation (in practice, ask user)
    user_mood = random.choice(['Q1', 'Q2', 'Q3', 'Q4']) 
    
    # Generate hybrid recommendations
    recs = recommend_songs_hybrid_eval(user_id, ref_track_name, user_item_sparse, user_factors, item_factors,
                                       music_info_metadata, knn_index, numerical_features, user_mood, n_recommendations=10)
    recommendations[user_id] = [t for t, s in recs]  # List of recommended track_ids

    # Generate pure CF recommendations (beta_cf=1, beta_cb=0, beta_mood=0)
    recs_cf = recommend_songs_hybrid_eval(user_id, ref_track_name, user_item_sparse, user_factors, item_factors,
                                    music_info_metadata, knn_index, numerical_features, user_mood, n_recommendations=10, beta_cf=1, beta_cb=0, beta_mood=0)
    recommendations_cf[user_id] = [t for t, s in recs_cf]

    # Generate pure CB recommendations (beta_cf=0, beta_cb=1, beta_mood=0)
    recs_cb = recommend_songs_hybrid_eval(user_id, ref_track_name, user_item_sparse, user_factors, item_factors,
                                    music_info_metadata, knn_index, numerical_features, user_mood, n_recommendations=10, beta_cf=0, beta_cb=1, beta_mood=0)
    recommendations_cb[user_id] = [t for t, s in recs_cb]

Processed 0 users
Processed 500 users
Processed 1000 users
Processed 1500 users
Processed 2000 users
Processed 2500 users
Processed 3000 users
Processed 3500 users
Processed 4000 users
Processed 4500 users
Processed 5000 users
Processed 5500 users
Processed 6000 users
Processed 6500 users
Processed 7000 users
Processed 7500 users
Processed 8000 users
Processed 8500 users
Processed 9000 users
Processed 9500 users
Processed 10000 users
Processed 10500 users
Processed 11000 users
Processed 11500 users
Processed 12000 users
Processed 12500 users
Processed 13000 users
Processed 13500 users
Processed 14000 users
Processed 14500 users
Processed 15000 users
Processed 15500 users
Processed 16000 users
Processed 16500 users
Processed 17000 users
Processed 17500 users
Processed 18000 users
Processed 18500 users
Processed 19000 users
Processed 19500 users
Processed 20000 users
Processed 20500 users
Processed 21000 users
Processed 21500 users
Processed 22000 users
Processed 22500 users


# Compute Metrics

recommended = list of items predicted by recommender system (ordered by rank).

relevant = set of items that are actually relevant to user (playcount).

k = how many top-ranked items we evaluate

In [ ]:
from sklearn.metrics import ndcg_score
import numpy as np

# ------- Define evaluation metrics -------

# Of the top-K recommended items, how many are relevant?
def precision_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    relevant_set = set(relevant)  # Ensure relevant is a set for intersection
    return len(set(rec_k) & relevant_set) / k if k > 0 else 0

# How many of the relevant items did we successfully recommend?
def recall_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    relevant_set = set(relevant)
    return len(set(rec_k) & relevant_set) / len(relevant_set) if relevant_set else 0

# Did we recommend at least one relevant item in top-K?
def hit_rate_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    relevant_set = set(relevant)
    return 1 if set(rec_k) & relevant_set else 0

# Normalized Discounted Cumulative Gain
# How well-ranked the relevant items are. Relevant items near the top contribute more.
def ndcg_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    if len(rec_k) == 0:  # Handle empty recommendations
        return 0
    relevant_set = set(relevant)
    y_true = [1 if t in relevant_set else 0 for t in rec_k]
    y_score = list(range(len(rec_k), 0, -1))  # Higher rank = higher score
    return ndcg_score([y_true], [y_score], k=k)

# Mean Average Precision
# Average Precision accross all relevant items, considering their rank in recommendations
def map_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    relevant_set = set(relevant)
    ap = 0
    num_relevant = 0
    for i, t in enumerate(rec_k):
        if t in relevant_set:
            num_relevant += 1
            ap += num_relevant / (i + 1)
    return ap / len(relevant_set) if relevant_set else 0




In [ ]:
# Evaluate recommendations at K=5
k = 5

# ---------------------------------
# Hybrid Recommendations Evaluation
# ---------------------------------

# Compute averages across users
precisions, recalls, hits, ndcgs, maps = [], [], [], [], []
for user_id, recs in recommendations.items():
    relevant = test_user_tracks.get(user_id, [])  
    if not relevant or not recs:  # Skip users with no test tracks or no recommendations
        continue
    precisions.append(precision_at_k(recs, relevant, k))
    recalls.append(recall_at_k(recs, relevant, k))
    hits.append(hit_rate_at_k(recs, relevant, k))
    ndcgs.append(ndcg_at_k(recs, relevant, k))
    maps.append(map_at_k(recs, relevant, k))

# Print results
print(f"(Hybrid) Average Precision@{k}: {np.mean(precisions):.4f}")
print(f"(Hybrid) Average Recall@{k}: {np.mean(recalls):.4f}")
print(f"(Hybrid) Average Hit Rate@{k}: {np.mean(hits):.4f}")
print(f"(Hybrid) Average NDCG@{k}: {np.mean(ndcgs):.4f}")
print(f"(Hybrid) Average MAP@{k}: {np.mean(maps):.4f}")
print("\n=============================\n")


# -----------------------------
# CF Recommendations Evaluation
# -----------------------------
precisions, recalls, hits, ndcgs, maps = [], [], [], [], []
for user_id, recs in recommendations_cf.items():
    relevant = test_user_tracks.get(user_id, [])  # This is a list
    if not relevant or not recs:  # Skip users with no test tracks or no recommendations
        continue
    precisions.append(precision_at_k(recs, relevant, k))
    recalls.append(recall_at_k(recs, relevant, k))
    hits.append(hit_rate_at_k(recs, relevant, k))
    ndcgs.append(ndcg_at_k(recs, relevant, k))
    maps.append(map_at_k(recs, relevant, k))

print(f"(CF) Average Precision@{k}: {np.mean(precisions):.4f}")
print(f"(CF) Average Recall@{k}: {np.mean(recalls):.4f}")
print(f"(CF) Average Hit Rate@{k}: {np.mean(hits):.4f}")
print(f"(CF) Average NDCG@{k}: {np.mean(ndcgs):.4f}")
print(f"(CF) Average MAP@{k}: {np.mean(maps):.4f}")
print("\n=============================\n")

# -----------------------------
# CB Recommendations Evaluation
# -----------------------------
precisions, recalls, hits, ndcgs, maps = [], [], [], [], []
for user_id, recs in recommendations_cb.items():
    relevant = test_user_tracks.get(user_id, [])  # This is a list
    if not relevant or not recs:  # Skip users with no test tracks or no recommendations
        continue
    precisions.append(precision_at_k(recs, relevant, k))
    recalls.append(recall_at_k(recs, relevant, k))
    hits.append(hit_rate_at_k(recs, relevant, k))
    ndcgs.append(ndcg_at_k(recs, relevant, k))
    maps.append(map_at_k(recs, relevant, k))

print(f"(CB) Average Precision@{k}: {np.mean(precisions):.4f}")
print(f"(CB) Average Recall@{k}: {np.mean(recalls):.4f}")
print(f"(CB) Average Hit Rate@{k}: {np.mean(hits):.4f}")
print(f"(CB) Average NDCG@{k}: {np.mean(ndcgs):.4f}")
print(f"(CB) Average MAP@{k}: {np.mean(maps):.4f}")
print("\n=============================\n")

(Hybrid) Average Precision@5: 0.0258
(Hybrid) Average Recall@5: 0.0088
(Hybrid) Average Hit Rate@5: 0.1125
(Hybrid) Average NDCG@5: 0.0754
(Hybrid) Average MAP@5: 0.0050


(CF) Average Precision@5: 0.0568
(CF) Average Recall@5: 0.0195
(CF) Average Hit Rate@5: 0.2264
(CF) Average NDCG@5: 0.1409
(CF) Average MAP@5: 0.0100


(CB) Average Precision@5: 0.0003
(CB) Average Recall@5: 0.0001
(CB) Average Hit Rate@5: 0.0015
(CB) Average NDCG@5: 0.0009
(CB) Average MAP@5: 0.0001


